# ⚙️ Data Preprocessing

## Objective

The objective of this notebook is to transform the cleaned emergency response dataset into a machine-learning ready dataset.

The following preprocessing techniques will be applied:

- Load cleaned dataset
- Verify dataset integrity
- Convert data types
- Convert date and time features
- Extract useful temporal features
- Encode categorical variables
- Feature scaling (if required)
- Save preprocessed dataset

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [ ]:
## 📂 Load Cleaned Dataset

The cleaned dataset generated from Notebook 2 is loaded for preprocessing.

In [2]:
clean_df = pd.read_csv("../data/processed/911_cleaned.csv")

print("✅ Cleaned dataset loaded successfully.")

✅ Cleaned dataset loaded successfully.


## 👀 Preview Dataset

In [3]:
clean_df.head()

,lat,lng,desc,zip,title,timeStamp,twp,addr,e
0,40.297876,-75.581294,REINDEER CT & DEAD END; NEW HANOVER; Station ...,19525.0,EMS: BACK PAINS/INJURY,2015-12-10 17:10:52,NEW HANOVER,REINDEER CT & DEAD END,1
1,40.258061,-75.264680,BRIAR PATH & WHITEMARSH LN; HATFIELD TOWNSHIP...,19446.0,EMS: DIABETIC EMERGENCY,2015-12-10 17:29:21,HATFIELD TOWNSHIP,BRIAR PATH & WHITEMARSH LN,1
2,40.121182,-75.351975,HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...,19401.0,Fire: GAS-ODOR/LEAK,2015-12-10 14:39:21,NORRISTOWN,HAWS AVE,1
3,40.116153,-75.343513,AIRY ST & SWEDE ST; NORRISTOWN; Station 308A;...,19401.0,EMS: CARDIAC EMERGENCY,2015-12-10 16:47:36,NORRISTOWN,AIRY ST & SWEDE ST,1
4,40.251492,-75.603350,CHERRYWOOD CT & DEAD END; LOWER POTTSGROVE; S...,Unknown,EMS: DIZZINESS,2015-12-10 16:56:52,LOWER POTTSGROVE,CHERRYWOOD CT & DEAD END,1


## 📋 Dataset Information

In [4]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 663282 entries, 0 to 663281
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   lat        663282 non-null  float64
 1   lng        663282 non-null  float64
 2   desc       663282 non-null  str    
 3   zip        663282 non-null  str    
 4   title      663282 non-null  str    
 5   timeStamp  663282 non-null  str    
 6   twp        663282 non-null  str    
 7   addr       663282 non-null  str    
 8   e          663282 non-null  int64  
dtypes: float64(2), int64(1), str(6)
memory usage: 45.5 MB


## 📑 Current Data Types

In [5]:
clean_df.dtypes.to_frame("Data Type")

,Data Type
lat,float64
lng,float64
desc,str
zip,str
title,str
timeStamp,str
twp,str
addr,str
e,int64


## 🕒 Convert Timestamp to Datetime

To perform time-based analysis, the `timeStamp` column must be converted from string format to the datetime datatype.

This allows extraction of useful temporal features such as year, month, day, hour, weekday, and weekend.

In [6]:
clean_df["timeStamp"] = pd.to_datetime(clean_df["timeStamp"])

print("✅ Timestamp converted successfully.")

✅ Timestamp converted successfully.


In [7]:
clean_df["timeStamp"].dtype

dtype('<M8[us]')

## 📅 Extract Temporal Features

Temporal features provide valuable information about emergency call patterns.

The following features are extracted:

- Year
- Month
- Day
- Hour
- Day of Week

In [8]:
clean_df["Year"] = clean_df["timeStamp"].dt.year
clean_df["Month"] = clean_df["timeStamp"].dt.month
clean_df["Day"] = clean_df["timeStamp"].dt.day
clean_df["Hour"] = clean_df["timeStamp"].dt.hour
clean_df["DayOfWeek"] = clean_df["timeStamp"].dt.day_name()

print("✅ Temporal features extracted.")

✅ Temporal features extracted.


In [9]:
clean_df[[
    "timeStamp",
    "Year",
    "Month",
    "Day",
    "Hour",
    "DayOfWeek"
]].head()

,timeStamp,Year,Month,Day,Hour,DayOfWeek
0,2015-12-10 17:10:52,2015,12,10,17,Thursday
1,2015-12-10 17:29:21,2015,12,10,17,Thursday
2,2015-12-10 14:39:21,2015,12,10,14,Thursday
3,2015-12-10 16:47:36,2015,12,10,16,Thursday
4,2015-12-10 16:56:52,2015,12,10,16,Thursday


## 🌤 Weekend Indicator

Create a binary feature indicating whether an emergency call occurred on a weekend.

In [10]:
clean_df["Weekend"] = clean_df["DayOfWeek"].isin(
    ["Saturday", "Sunday"]
)

In [11]:
clean_df[["DayOfWeek","Weekend"]].head(10)

,DayOfWeek,Weekend
0,Thursday,False
1,Thursday,False
2,Thursday,False
3,Thursday,False
4,Thursday,False
5,Thursday,False
6,Thursday,False
7,Thursday,False
8,Thursday,False
9,Thursday,False


## 🌅 Time of Day

Emergency calls are categorized into different periods of the day.

Categories:

- Night
- Morning
- Afternoon
- Evening

In [12]:
def time_of_day(hour):
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

clean_df["TimeOfDay"] = clean_df["Hour"].apply(time_of_day)

print("✅ TimeOfDay feature created.")

✅ TimeOfDay feature created.


In [13]:
clean_df[["Hour","TimeOfDay"]].head(15)

,Hour,TimeOfDay
0,17,Afternoon
1,17,Afternoon
2,14,Afternoon
3,16,Afternoon
4,16,Afternoon
5,15,Afternoon
6,16,Afternoon
7,16,Afternoon
8,16,Afternoon
9,17,Afternoon


## 🚨 Emergency Category

Extract the primary emergency category from the `title` column.

In [14]:
clean_df["Category"] = clean_df["title"].str.split(":").str[0]

print("✅ Emergency categories extracted.")

✅ Emergency categories extracted.


In [15]:
clean_df["Category"].value_counts()

Category
EMS        332591
Traffic    230115
Fire       100576
Name: count, dtype: int64

## 🔢 Label Encoding

Machine learning algorithms require numerical representations of categorical variables.

The emergency category is converted into numerical labels.

In [16]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

clean_df["CategoryEncoded"] = encoder.fit_transform(
    clean_df["Category"]
)

In [17]:
clean_df[[
    "Category",
    "CategoryEncoded"
]].head()

,Category,CategoryEncoded
0,EMS,0
1,EMS,0
2,Fire,1
3,EMS,0
4,EMS,0


In [18]:
clean_df.to_csv(
    "../data/processed/911_preprocessed.csv",
    index=False
)

print("✅ Preprocessed dataset saved successfully.")

✅ Preprocessed dataset saved successfully.


In [19]:
summary = pd.DataFrame({
    "Technique":[
        "Datetime Conversion",
        "Temporal Feature Extraction",
        "Weekend Feature",
        "Time Of Day",
        "Category Extraction",
        "Label Encoding"
    ],
    "Status":[
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed"
    ]
})

summary

,Technique,Status
0,Datetime Conversion,Completed
1,Temporal Feature Extraction,Completed
2,Weekend Feature,Completed
3,Time Of Day,Completed
4,Category Extraction,Completed
5,Label Encoding,Completed


## 📏 Feature Scaling

Machine learning algorithms often perform better when numerical features are on a similar scale.

In this project, the geographical coordinates (Latitude and Longitude) are standardized using StandardScaler.

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

clean_df[["Latitude_Scaled","Longitude_Scaled"]] = scaler.fit_transform(
    clean_df[["lat","lng"]]
)

print("✅ Numerical features scaled successfully.")

✅ Numerical features scaled successfully.


In [21]:
clean_df[
    ["lat","Latitude_Scaled","lng","Longitude_Scaled"]
].head()

,lat,Latitude_Scaled,lng,Longitude_Scaled
0,40.297876,0.633100,-75.581294,-0.168057
1,40.258061,0.452678,-75.264680,0.021171
2,40.121182,-0.167597,-75.351975,-0.031002
3,40.116153,-0.190386,-75.343513,-0.025944
4,40.251492,0.422909,-75.603350,-0.181239


## 📊 Emergency Category Distribution

Display the frequency of each emergency category.

In [22]:
clean_df["Category"].value_counts().to_frame("Count")

,Count
Category,
EMS,332591
Traffic,230115
Fire,100576


In [23]:
clean_df.head()

,lat,lng,desc,zip,title,timeStamp,twp,addr,e,Year,Month,Day,Hour,DayOfWeek,Weekend,TimeOfDay,Category,CategoryEncoded,Latitude_Scaled,Longitude_Scaled
0,40.297876,-75.581294,REINDEER CT & DEAD END; NEW HANOVER; Station ...,19525.0,EMS: BACK PAINS/INJURY,2015-12-10 17:10:52,NEW HANOVER,REINDEER CT & DEAD END,1,2015,12,10,17,Thursday,False,Afternoon,EMS,0,0.633100,-0.168057
1,40.258061,-75.264680,BRIAR PATH & WHITEMARSH LN; HATFIELD TOWNSHIP...,19446.0,EMS: DIABETIC EMERGENCY,2015-12-10 17:29:21,HATFIELD TOWNSHIP,BRIAR PATH & WHITEMARSH LN,1,2015,12,10,17,Thursday,False,Afternoon,EMS,0,0.452678,0.021171
2,40.121182,-75.351975,HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...,19401.0,Fire: GAS-ODOR/LEAK,2015-12-10 14:39:21,NORRISTOWN,HAWS AVE,1,2015,12,10,14,Thursday,False,Afternoon,Fire,1,-0.167597,-0.031002
3,40.116153,-75.343513,AIRY ST & SWEDE ST; NORRISTOWN; Station 308A;...,19401.0,EMS: CARDIAC EMERGENCY,2015-12-10 16:47:36,NORRISTOWN,AIRY ST & SWEDE ST,1,2015,12,10,16,Thursday,False,Afternoon,EMS,0,-0.190386,-0.025944
4,40.251492,-75.603350,CHERRYWOOD CT & DEAD END; LOWER POTTSGROVE; S...,Unknown,EMS: DIZZINESS,2015-12-10 16:56:52,LOWER POTTSGROVE,CHERRYWOOD CT & DEAD END,1,2015,12,10,16,Thursday,False,Afternoon,EMS,0,0.422909,-0.181239


In [24]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 663282 entries, 0 to 663281
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   lat               663282 non-null  float64       
 1   lng               663282 non-null  float64       
 2   desc              663282 non-null  str           
 3   zip               663282 non-null  str           
 4   title             663282 non-null  str           
 5   timeStamp         663282 non-null  datetime64[us]
 6   twp               663282 non-null  str           
 7   addr              663282 non-null  str           
 8   e                 663282 non-null  int64         
 9   Year              663282 non-null  int32         
 10  Month             663282 non-null  int32         
 11  Day               663282 non-null  int32         
 12  Hour              663282 non-null  int32         
 13  DayOfWeek         663282 non-null  str           
 14  Weekend        

In [25]:
print("="*50)
print("PREPROCESSED DATASET")
print("="*50)

print(f"Rows : {clean_df.shape[0]:,}")
print(f"Columns : {clean_df.shape[1]}")

PREPROCESSED DATASET
Rows : 663,282
Columns : 20


In [26]:
clean_df.to_csv(
    "../data/processed/911_preprocessed.csv",
    index=False
)

print("✅ Preprocessed dataset saved.")

✅ Preprocessed dataset saved.


In [28]:
summary = pd.DataFrame({
    "Step":[
        "Missing Value Treatment",
        "Duplicate Removal",
        "Datetime Conversion",
        "Temporal Features",
        "Weekend Feature",
        "Time of Day",
        "Category Extraction",
        "Label Encoding",
        "Feature Scaling",
        "Dataset Export"
    ],
    "Status":["Completed"]*10
})

summary

,Step,Status
0,Missing Value Treatment,Completed
1,Duplicate Removal,Completed
2,Datetime Conversion,Completed
3,Temporal Features,Completed
4,Weekend Feature,Completed
5,Time of Day,Completed
6,Category Extraction,Completed
7,Label Encoding,Completed
8,Feature Scaling,Completed
9,Dataset Export,Completed


# ✅ Conclusion

The emergency response dataset has now been successfully preprocessed.

### Preprocessing Techniques Applied

- Missing value treatment
- Duplicate removal
- Datetime conversion
- Temporal feature extraction
- Weekend feature creation
- Time of Day categorization
- Emergency category extraction
- Label encoding
- Feature scaling
- Dataset export

The dataset is now suitable for exploratory data analysis and future machine learning applications.